In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import joblib

# ------------------------------------
# 1. CHARGEMENT DU DATASET
# ------------------------------------
DATA_PATH = "dataset_pneumonie_simule.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset chargé ✅")
print(df.shape)
print(df.head())

# ------------------------------------
# 2. VÉRIFICATIONS DE BASE
# ------------------------------------
assert "patient_id" in df.columns
assert "pneumonia_t_plus_72h" in df.columns

print("\nTaux de pneumonie :")
print(df["pneumonia_t_plus_72h"].mean())

# ------------------------------------
# 3. SPLIT TRAIN / TEST PAR PATIENT
# (PAS par ligne → évite data leakage)
# ------------------------------------
patient_ids = df["patient_id"].unique()

train_patients, test_patients = train_test_split(
    patient_ids,
    test_size=0.2,
    random_state=42
)

train_df = df[df["patient_id"].isin(train_patients)]
test_df = df[df["patient_id"].isin(test_patients)]

print("\nPatients train :", len(train_patients))
print("Patients test  :", len(test_patients))

# ------------------------------------
# 4. SÉPARATION FEATURES / LABEL
# ------------------------------------
TARGET = "pneumonia_t_plus_72h"

# Colonnes À EXCLURE du modèle
EXCLUDE_COLS = [
    "patient_id",
    "time_index",
    TARGET
]

FEATURES = [c for c in df.columns if c not in EXCLUDE_COLS]

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

print("\nNombre de features :", len(FEATURES))

# ------------------------------------
# 5. PIPELINE DE MODÈLE
# ------------------------------------
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        scale_pos_weight=(
            len(y_train[y_train == 0]) /
            len(y_train[y_train == 1])
        ),
        random_state=42
    ))
])

# ------------------------------------
# 6. ENTRAÎNEMENT
# ------------------------------------
print("\nEntraînement du modèle...")
model.fit(X_train, y_train)
print("Entraînement terminé ✅")

# ------------------------------------
# 7. ÉVALUATION
# ------------------------------------
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)

print("\nAUC ROC :", round(auc, 3))
print("\nRapport de classification :")
print(classification_report(y_test, y_pred))

print("\nMatrice de confusion :")
print(confusion_matrix(y_test, y_pred))

# ------------------------------------
# 8. SAUVEGARDE DU MODÈLE
# ------------------------------------
MODEL_PATH = "modele_pneumonie_xgb_gptcsv.pkl"
joblib.dump(model, MODEL_PATH)

print(f"\nModèle sauvegardé ✅ → {MODEL_PATH}")


Dataset chargé ✅
(10124, 20)
   patient_id  time_index  age  smoking  diabetes  copd_asthma  \
0           0           0   69        1         0            0   
1           0           1   69        1         0            0   
2           0           2   69        1         0            0   
3           0           3   69        1         0            0   
4           0           4   69        1         0            0   

   immunosuppression  temperature  respiratory_rate  heart_rate  spo2  \
0                  0        37.19                17          70  98.5   
1                  0        37.15                17          72  98.3   
2                  0        37.14                17          72  98.0   
3                  0        37.13                18          72  98.0   
4                  0        37.21                18          72  97.8   

   systolic_bp   wbc  curb65  pneumonia_t_plus_72h  delta_respiratory_rate  \
0          121  6695       1                     0       

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# --- Prédictions sur ton jeu de test ---
y_pred = model.predict(X_test)

# --- Calcul des métriques ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("📊 Performance du modèle XGBoost")
print(f"Accuracy  : {accuracy:.3f}")
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"F1-score  : {f1:.3f}")

📊 Performance du modèle XGBoost
Accuracy  : 0.993
Precision : 0.887
Recall    : 0.956
F1-score  : 0.920
